# 🏃 Personal Running Coach Agent Swarm
## Phase 3 — ChromaDB Memory Ingestion

**Project:** Agentic AI Course — Will Sutherland, May 2026

Builds and populates the three ChromaDB collections that all agents
will query for context during Phase 4.

### Collections

| Collection | Content | Updated |
|---|---|---|
| `workout_summaries` | Natural language summary of every run from the last 6 months | Re-run after each Strava/Garmin refresh |
| `athlete_profile` | Static athlete background, goals, training context | Edit manually when profile changes |
| `session_notes` | Free-text notes added after key sessions | Add via the notes cell at the bottom |

### This notebook covers
1. Setup — mount Drive, imports, connect to ChromaDB
2. Build `workout_summaries` collection
3. Build `athlete_profile` collection
4. Build `session_notes` collection
5. Build retrieval wrapper used by all agents
6. Smoke tests — verify retrieval is working
7. Add session notes (run this cell after key workouts)

---
## 1. Setup

In [2]:
import os, sys, json, sqlite3
from datetime import date, timedelta
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)

BASE_DIR       = "/content/drive/MyDrive/running_coach"
WORKOUTS_PATH  = f"{BASE_DIR}/data/processed/workouts_normalized.json"
GARMIN_DB_PATH = f"{BASE_DIR}/data/raw/garmin/garmin.db"
CHROMA_DIR     = f"{BASE_DIR}/memory/chroma"

sys.path.insert(0, f"{BASE_DIR}/tools")

GEMINI_API_KEY = userdata.get('key')

%pip install -q chromadb google-generativeai

import chromadb
from chromadb.utils import embedding_functions
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

# ── ChromaDB client (persisted to Drive) ──────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# ── Embedding function — Gemini gemini-embedding-001 ────────────────────────
gemini_ef = embedding_functions.GoogleGenerativeAiEmbeddingFunction(
    api_key=GEMINI_API_KEY,
    model_name="models/gemini-embedding-001",
)

print("✅ Setup complete.")
print(f"   ChromaDB path:    {CHROMA_DIR}")
print(f"   Workouts file:    {os.path.exists(WORKOUTS_PATH)}")
print(f"   Garmin DB:        {os.path.exists(GARMIN_DB_PATH)}")
print(f"   Existing collections: {[c.name for c in chroma_client.list_collections()]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
E

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Setup complete.
   ChromaDB path:    /content/drive/MyDrive/running_coach/memory/chroma
   Workouts file:    True
   Garmin DB:        True
   Existing collections: ['workout_summaries', 'athlete_profile', 'session_notes']


---
## 2. `workout_summaries` Collection

Each WorkoutRecord is converted to a natural language summary chunk and
embedded into ChromaDB. Agents retrieve the most semantically relevant
past sessions when answering questions about training history.

**Chunk format:**
```
Date: 2026-04-22 | Type: threshold | Distance: 12.3km | Duration: 49.5min
Pace: 4:03/km (target: 4:01/km, 2s slower) | HR: 162 avg / 178 max
Training load: 187 | Aerobic effect: 3.8 | Suffer score: 67
Recovery context: HRV 95 (balanced), sleep 7.8h, body battery at wake: 62, readiness: 74
```

> **Safe to re-run** — deletes and rebuilds the collection each time
> so you always have a fresh, consistent index after a data refresh.

In [3]:
from parse_workout_data import load_workouts, format_pace
from calculate_pace_zones import classify_workouts
from query_garmin_db import get_activity_splits

# ── Load and classify workouts ─────────────────────────────────────────────────
workouts  = load_workouts(WORKOUTS_PATH, days=180)
enriched  = classify_workouts(workouts)
print(f"Loaded {len(enriched)} workouts for ingestion.")


def get_splits_summary(date_str: str, conn) -> str | None:
    """
    Pull lap splits for a given date from Garmin DB and summarise
    hard efforts vs easy/recovery laps for embedding.
    Returns a one-line summary string, or None if no splits available.
    """
    try:
        row = conn.execute("""
            SELECT activity_id FROM activity
            WHERE DATE(start_time_local) = ?
              AND LOWER(activity_type) LIKE '%run%'
            ORDER BY start_time_local DESC LIMIT 1
        """, (date_str,)).fetchone()
        if not row:
            return None
        splits = get_activity_splits(conn, row[0])
        if not splits:
            return None

        # Compute session average pace
        full_km = [s for s in splits if s['distance_m'] >= 900]
        if not full_km:
            return None
        avg_s_per_km = sum(
            s['duration_s'] / (s['distance_m'] / 1000) for s in full_km
        ) / len(full_km)

        # Classify hard vs easy by >15s/km faster than average
        hard = [s for s in splits if s['distance_m'] >= 200 and
                s['duration_s'] / (s['distance_m'] / 1000) < avg_s_per_km - 15]
        easy = [s for s in splits if s['distance_m'] >= 200 and
                s['duration_s'] / (s['distance_m'] / 1000) > avg_s_per_km + 15]

        if not hard:
            return None

        # Compute hard effort pace stats
        hard_paces = [
            s['duration_s'] / (s['distance_m'] / 1000) / 60
            for s in hard
        ]
        avg_hard = sum(hard_paces) / len(hard_paces)
        min_hard = min(hard_paces)

        def fmt(p):
            m = int(p); s = round((p - m) * 60)
            return f"{m}:{s:02d}/km"

        parts = [f"Hard efforts: {len(hard)} reps, avg {fmt(avg_hard)}, fastest {fmt(min_hard)}"]
        if easy:
            easy_paces = [
                s['duration_s'] / (s['distance_m'] / 1000) / 60
                for s in easy
            ]
            avg_easy = sum(easy_paces) / len(easy_paces)
            parts.append(f"Recovery/easy laps: {len(easy)}, avg {fmt(avg_easy)}")

        return ' | '.join(parts)
    except Exception:
        return None


def build_workout_chunk(w: dict, conn=None) -> str:
    """
    Convert a classified WorkoutRecord to a natural language summary
    suitable for semantic embedding and retrieval.
    For structured sessions, includes a split summary of hard effort paces
    so the embedding captures interval intensity, not just overall average pace.
    """
    cl = w.get('classification', {})
    pe = w.get('pace_evaluation')
    hc = w.get('health_context', {})

    workout_type = cl.get('workout_type') or 'unclassified'
    pace_str     = format_pace(w.get('avg_pace_min_km'))

    # Pace evaluation line
    if pe and pe.get('verdict'):
        pace_line = f"Overall avg pace: {pace_str} | {pe['verdict']}"
    else:
        pace_line = f"Overall avg pace: {pace_str}"

    # Health context line
    hc_parts = []
    if hc.get('hrv_status'):
        hc_parts.append(f"HRV status: {hc['hrv_status']}")
    if hc.get('hrv_last_night'):
        hc_parts.append(f"HRV last night: {hc['hrv_last_night']}")
    if hc.get('sleep_sleep_time_seconds'):
        hrs = round(hc['sleep_sleep_time_seconds'] / 3600, 1)
        hc_parts.append(f"sleep: {hrs}h")
    if hc.get('body_battery_at_wake'):
        hc_parts.append(f"body battery at wake: {hc['body_battery_at_wake']}")
    if hc.get('training_readiness_score'):
        hc_parts.append(f"readiness: {hc['training_readiness_score']}")
    if hc.get('heart_rate_resting_hr'):
        hc_parts.append(f"resting HR: {hc['heart_rate_resting_hr']} bpm")

    hc_line = f"Recovery context: {', '.join(hc_parts)}" if hc_parts else ""

    lines = [
        f"Date: {w.get('date')} | Type: {workout_type} | "
        f"Distance: {w.get('distance_km')}km | Duration: {w.get('duration_min')}min",
        pace_line,
        f"HR: {w.get('avg_hr')} avg / {w.get('max_hr')} max | "
        f"Elevation: {w.get('elevation_m')}m",
        f"Training load: {w.get('training_load')} | "
        f"Aerobic effect: {w.get('aerobic_effect')} | "
        f"Anaerobic effect: {w.get('anaerobic_effect')} | "
        f"Suffer score: {w.get('suffer_score')}",
    ]

    # Add split summary for structured sessions
    if conn and workout_type not in ('easy', 'unclassified'):
        splits_summary = get_splits_summary(w.get('date', ''), conn)
        if splits_summary:
            lines.append(f"Interval splits: {splits_summary}")

    if hc_line:
        lines.append(hc_line)
    if w.get('garmin_enriched'):
        lines.append("Garmin enriched: yes")

    return "\n".join(lines)


# ── Build collection ───────────────────────────────────────────────────────────
# Delete and recreate for a clean rebuild
import sqlite3 as _sqlite3
garmin_conn = _sqlite3.connect(GARMIN_DB_PATH)
try:
    chroma_client.delete_collection("workout_summaries")
    print("Deleted existing workout_summaries collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="workout_summaries",
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

# Ingest in batches of 10 (Gemini embedding API rate limit)
import time
BATCH_SIZE = 10
documents, ids, metadatas = [], [], []

for w in enriched:
    chunk = build_workout_chunk(w, conn=garmin_conn)
    documents.append(chunk)
    ids.append(w['activity_id'])
    metadatas.append({
        'date':         w.get('date', ''),
        'workout_type': w.get('classification', {}).get('workout_type') or 'unknown',
        'distance_km':  str(w.get('distance_km', '')),
        'garmin_enriched': str(w.get('garmin_enriched', False)),
        'source':       w.get('source', ''),
    })

total    = len(documents)
ingested = 0
for i in range(0, total, BATCH_SIZE):
    batch_docs  = documents[i:i+BATCH_SIZE]
    batch_ids   = ids[i:i+BATCH_SIZE]
    batch_meta  = metadatas[i:i+BATCH_SIZE]
    collection.add(documents=batch_docs, ids=batch_ids, metadatas=batch_meta)
    ingested += len(batch_docs)
    print(f"  Ingested {ingested}/{total}...")
    time.sleep(1)  # Respect Gemini API rate limit

garmin_conn.close()
print(f"\n✅ workout_summaries: {collection.count()} documents ingested.")

Loaded 115 workouts for ingestion.
Deleted existing workout_summaries collection.
  Ingested 10/115...
  Ingested 20/115...
  Ingested 30/115...
  Ingested 40/115...
  Ingested 50/115...
  Ingested 60/115...
  Ingested 70/115...
  Ingested 80/115...
  Ingested 90/115...
  Ingested 100/115...
  Ingested 110/115...
  Ingested 115/115...

✅ workout_summaries: 115 documents ingested.


---
## 3. `athlete_profile` Collection

Static document describing Will's training background, current block,
goals, and coaching context. Agents retrieve this for personalisation.
Edit the profile dict below when anything changes (e.g. after the race).

> **Safe to re-run** — rebuilds the profile document each time.

In [ ]:
# ── Athlete profile document ───────────────────────────────────────────────────
# Edit this dict to update the profile. Re-run the cell to push changes.

ATHLETE_PROFILE = {
    "name":             "Will Sutherland",
    "location":         "Halifax, Nova Scotia",
    "running_since":    "Fall 2023",
    "experience":       "Approximately 2.5 years of serious training",

    "race_history": [
        {"event": "Half Marathon", "time": "1:27:50"},
        {"event": "Half Marathon", "time": "1:28:xx"},
        {"event": "Full Marathon", "time": "3:09", "notes": "Strong race"},
        {"event": "Full Marathon", "time": "3:19",
         "notes": "Injured, walked last 12km. Was on 4:15/km pace through 30km."},
    ],

    "current_race": {
        "name":    "Fredericton Half Marathon",
        "date":    "2026-05-10",
        "goal_a":  "1:23:59 (A+ goal)",
        "goal_b":  "Sub 1:25:00 (happy with this)",
        "race_strategy": "Negative split — conservative first half, build in second half",
    },

    "current_block": {
        "plan_length":    "18 weeks total (16 weeks build + 2 week taper)",
        "training_days":  "5 days per week: Tuesday, Wednesday, Thursday, Saturday, Sunday",
        "rest_days":      "Monday and Friday",
        "typical_week": [
            {"day": "Tuesday",   "session": "Easy run or fartlek"},
            {"day": "Wednesday", "session": "Speed workout (key session) — e.g. 10x1km at HM pace"},
            {"day": "Thursday",  "session": "Easy run"},
            {"day": "Saturday",  "session": "Long run with structure — warm up/cool down at MP+10s, middle at HM pace"},
            {"day": "Sunday",    "session": "Easy run"},
        ],
    },

    "training_paces": {
        "easy":      "By feel — anything slower than 4:45/km",
        "marathon":  "4:14/km",
        "threshold": "4:01/km",
        "1hr":       "3:56/km",
        "fartlek":   "3:49/km",
        "8k":        "3:46/km",
        "vo2max":    "3:40/km",
        "hm_race":   "~3:59/km (for 1:24 target)",
    },

    "devices":          "Garmin Forerunner 265",
    "data_sources":     "Garmin Connect (via garmin-givemydata), Strava",

    "injuries": {
        "current":   "None",
        "history":   "Previous marathon DNF-equivalent due to injury at 30km mark",
        "watch_areas": "Monitor for general overtraining signs given training volume",
    },

    "coaching_context": (
        "Will trains under a coach on a structured 18-week periodized plan. "
        "The Wednesday speed session and Saturday long run are the key quality sessions each week. "
        "Easy runs should be genuinely easy — by feel, no pace target, ceiling at 4:45/km. "
        "The 10% mileage rule applies week-on-week. "
        "Injury disclaimer: any injury-related queries should be redirected to Will's coach or a physio."
    ),

    "agent_notes": (
        "When evaluating workouts, always compare against coach-prescribed targets, not generic benchmarks. "
        "Pace tolerance is +/- 5 seconds from target. "
        "Recovery decisions should weight: training readiness score, HRV status, body battery at wake, and TSB. "
        "Race predictions from Garmin suggest ~1:23 half marathon fitness — consistent with goal."
    ),
}


def profile_to_text(profile: dict) -> str:
    """Convert the athlete profile dict to a flat text document for embedding."""
    lines = [
        f"Athlete: {profile['name']}, {profile['location']}",
        f"Running since: {profile['running_since']} ({profile['experience']})",
        "",
        "Race history:",
    ]
    for r in profile['race_history']:
        note = f" ({r['notes']})" if r.get('notes') else ""
        lines.append(f"  {r['event']}: {r['time']}{note}")

    cr = profile['current_race']
    lines += [
        "",
        f"Target race: {cr['name']} on {cr['date']}",
        f"Goal A: {cr['goal_a']}",
        f"Goal B: {cr['goal_b']}",
        f"Race strategy: {cr['race_strategy']}",
        "",
        f"Training plan: {profile['current_block']['plan_length']}",
        f"Training days: {profile['current_block']['training_days']}",
        f"Rest days: {profile['current_block']['rest_days']}",
        "",
        "Typical weekly structure:",
    ]
    for day in profile['current_block']['typical_week']:
        lines.append(f"  {day['day']}: {day['session']}")

    lines += [
        "",
        "Training paces (coach-prescribed):",
    ]
    for pace_type, pace in profile['training_paces'].items():
        lines.append(f"  {pace_type}: {pace}")

    inj = profile['injuries']
    lines += [
        "",
        f"Current injuries: {inj['current']}",
        f"Injury history: {inj['history']}",
        f"Watch areas: {inj['watch_areas']}",
        "",
        f"Coaching context: {profile['coaching_context']}",
        "",
        f"Agent notes: {profile['agent_notes']}",
    ]
    return "\n".join(lines)


profile_text = profile_to_text(ATHLETE_PROFILE)

# ── Build collection ───────────────────────────────────────────────────────────
try:
    chroma_client.delete_collection("athlete_profile")
except Exception:
    pass

profile_col = chroma_client.create_collection(
    name="athlete_profile",
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

# Split into paragraphs for finer-grained retrieval
paragraphs = [p.strip() for p in profile_text.split('\n\n') if p.strip()]
profile_col.add(
    documents=paragraphs,
    ids=[f"profile_{i}" for i in range(len(paragraphs))],
    metadatas=[{"section": "athlete_profile", "chunk": i} for i in range(len(paragraphs))]
)

print(f"✅ athlete_profile: {profile_col.count()} chunks ingested.")
print("\nProfile document:")
print(profile_text)

---
## 4. `session_notes` Collection

Free-text notes added after key sessions. These give agents qualitative
context alongside the numbers — how the run felt, any niggles, weather,
mental state, etc.

**To add a note:** scroll to Section 7 at the bottom and fill in the
date and note text, then run that cell.

This cell creates the collection if it doesn't exist. It does **not**
wipe existing notes — notes accumulate over time.

In [ ]:
# Get or create — does NOT delete existing notes
notes_col = chroma_client.get_or_create_collection(
    name="session_notes",
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ session_notes collection ready: {notes_col.count()} note(s) stored.")
if notes_col.count() > 0:
    existing = notes_col.get()
    print("\nExisting notes:")
    for doc, meta in zip(existing['documents'], existing['metadatas']):
        print(f"  [{meta.get('date', 'unknown')}] {doc[:120]}{'...' if len(doc) > 120 else ''}")

---
## 5. Retrieval Wrapper

A single function that all agents call to query memory.
Writes `memory_retrieval.py` to `tools/` on Drive.

In [ ]:
retrieval_code = '''
"""
tools/memory_retrieval.py

Retrieval wrapper for the ChromaDB vector store.
All agents call these functions — no agent queries ChromaDB directly.

Collections
-----------
workout_summaries  — natural language summaries of recent runs
athlete_profile    — static athlete background and coaching context
session_notes      — free-text post-session notes

Public API
----------
get_client(chroma_dir, api_key)     -> tuple[chroma client, embedding fn]
retrieve_workouts(client, ef, query, n, filters)  -> list[dict]
retrieve_profile(client, ef, query, n)            -> str
retrieve_notes(client, ef, query, n)              -> list[dict]
retrieve_all(client, ef, query, n)                -> dict
format_memory_context(results)                    -> str
"""

from __future__ import annotations
import chromadb
from chromadb.utils import embedding_functions


def get_client(
    chroma_dir: str,
    api_key: str,
) -> tuple:
    """
    Initialize and return (chroma_client, embedding_function).
    Call once per agent session and pass the results to retrieve_* functions.
    """
    client = chromadb.PersistentClient(path=chroma_dir)
    ef = embedding_functions.GoogleGenerativeAiEmbeddingFunction(
        api_key=api_key,
        model_name="models/gemini-embedding-001",
    )
    return client, ef


def retrieve_workouts(
    client,
    ef,
    query: str,
    n: int = 5,
    filters: dict | None = None,
) -> list[dict]:
    """
    Retrieve the N most relevant workout summaries for a query.

    Parameters
    ----------
    query   : Natural language query e.g. "recent threshold sessions"
    n       : Number of results to return
    filters : Optional ChromaDB where clause e.g. {"workout_type": "threshold"}

    Returns list of dicts with keys: document, metadata, distance
    """
    try:
        col = client.get_collection("workout_summaries", embedding_function=ef)
    except Exception:
        return []

    kwargs = {"query_texts": [query], "n_results": min(n, col.count())}
    if filters:
        kwargs["where"] = filters

    results = col.query(**kwargs)

    return [
        {
            "document": doc,
            "metadata": meta,
            "distance": dist,
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def retrieve_profile(
    client,
    ef,
    query: str = "athlete background goals training context",
    n: int = 5,
) -> str:
    """
    Retrieve the most relevant athlete profile chunks for a query.
    Returns a single concatenated string for prompt injection.
    """
    try:
        col = client.get_collection("athlete_profile", embedding_function=ef)
    except Exception:
        return ""

    results = col.query(
        query_texts=[query],
        n_results=min(n, col.count())
    )
    return "\n\n".join(results["documents"][0])


def retrieve_notes(
    client,
    ef,
    query: str,
    n: int = 3,
) -> list[dict]:
    """
    Retrieve the N most relevant session notes for a query.
    Returns list of dicts with keys: document, metadata, distance
    """
    try:
        col = client.get_collection("session_notes", embedding_function=ef)
        if col.count() == 0:
            return []
    except Exception:
        return []

    results = col.query(
        query_texts=[query],
        n_results=min(n, col.count())
    )
    return [
        {
            "document": doc,
            "metadata": meta,
            "distance": dist,
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


def retrieve_all(
    client,
    ef,
    query: str,
    n_workouts: int = 5,
    n_profile:  int = 4,
    n_notes:    int = 3,
) -> dict:
    """
    Retrieve from all three collections in one call.
    Returns dict with keys: workouts, profile, notes.
    Used by the Coordinator to assemble full context.
    """
    return {
        "workouts": retrieve_workouts(client, ef, query, n=n_workouts),
        "profile":  retrieve_profile(client, ef, query, n=n_profile),
        "notes":    retrieve_notes(client, ef, query, n=n_notes),
    }


def format_memory_context(results: dict) -> str:
    """
    Format retrieve_all() results into a plain-text block
    ready for injection into an agent prompt.
    """
    lines = ["=== Memory Context ==="]

    if results.get("profile"):
        lines += ["\n--- Athlete Profile ---", results["profile"]]

    if results.get("workouts"):
        lines.append("\n--- Relevant Past Workouts ---")
        for r in results["workouts"]:
            lines.append(r["document"])
            lines.append("")

    if results.get("notes"):
        lines.append("--- Session Notes ---")
        for r in results["notes"]:
            date = r["metadata"].get("date", "unknown")
            lines.append(f"[{date}] {r['document']}")

    return "\n".join(lines)
'''

tool_path = os.path.join(BASE_DIR, "tools", "memory_retrieval.py")
with open(tool_path, 'w') as f:
    f.write(retrieval_code.strip())

print(f"✅ memory_retrieval.py written to {tool_path}")

---
## 6. Smoke Tests

In [ ]:
print("=" * 60)
print("TEST 6.1 — Collection counts")
print("=" * 60)
for name in ['workout_summaries', 'athlete_profile', 'session_notes']:
    try:
        col   = chroma_client.get_collection(name, embedding_function=gemini_ef)
        count = col.count()
        status = '✅' if count > 0 else '⚠️  empty'
        print(f"  {status} {name}: {count} documents")
    except Exception as e:
        print(f"  ❌ {name}: {e}")

In [ ]:
print("=" * 60)
print("TEST 6.2 — Workout retrieval")
print("=" * 60)

from memory_retrieval import get_client, retrieve_workouts, retrieve_profile, retrieve_all, format_memory_context

mem_client, mem_ef = get_client(CHROMA_DIR, GEMINI_API_KEY)

test_queries = [
    "recent threshold sessions",
    "long run at marathon pace",
    "easy recovery run",
]

for query in test_queries:
    results = retrieve_workouts(mem_client, mem_ef, query, n=2)
    print(f"\nQuery: '{query}' → {len(results)} results")
    for r in results:
        first_line = r['document'].split('\n')[0]
        print(f"  [{r['metadata'].get('date')}] {first_line} (dist: {r['distance']:.3f})")

In [ ]:
print("=" * 60)
print("TEST 6.3 — Profile retrieval")
print("=" * 60)

profile_result = retrieve_profile(
    mem_client, mem_ef,
    query="athlete goals race target pace strategy",
    n=3
)
print(profile_result)
print("\n✅ Profile retrieval working.")

In [ ]:
print("=" * 60)
print("TEST 6.4 — Full memory context (as agents will see it)")
print("=" * 60)

results = retrieve_all(
    mem_client, mem_ef,
    query="how did my last threshold session go?",
    n_workouts=3,
    n_profile=2,
    n_notes=2,
)
context = format_memory_context(results)
print(context)
print("\n✅ Full memory context assembled correctly.")

---
## 7. Add Session Notes

Run this cell after any key session to add a qualitative note.
Notes are stored permanently in ChromaDB and retrieved by agents
when relevant to a query.

**Tips for good notes:**
- How did the effort feel vs the pace target?
- Any physical feedback (legs, breathing, niggles)?
- External factors (weather, sleep the night before, stress)?
- Did you hit the workout as prescribed or modify it?

In [ ]:
# ── Edit these two fields and run the cell ─────────────────────────────────────
NOTE_DATE = "2026-05-02"   # YYYY-MM-DD — date of the session
NOTE_TEXT = """            # Your note here — no length limit
Replace this with your actual note. For example:
Threshold workout, 5x2km. Legs felt heavy from Saturday's long run.
Hit 4:02/km on reps 1-3, drifted to 4:05 on reps 4-5. HR spiked higher
than usual on rep 4 (178 vs normal 172). Cooled down feeling okay.
Might need an extra easy day before Saturday.
"""
# ─────────────────────────────────────────────────────────────────────────────

note_text = NOTE_TEXT.strip()
if not note_text or note_text.startswith("Replace this"):
    print("⚠️  Please edit NOTE_DATE and NOTE_TEXT before running this cell.")
else:
    note_id = f"note_{NOTE_DATE}_{abs(hash(note_text)) % 10000:04d}"
    notes_col = chroma_client.get_or_create_collection(
        name="session_notes", embedding_function=gemini_ef
    )
    notes_col.add(
        documents=[note_text],
        ids=[note_id],
        metadatas=[{"date": NOTE_DATE, "added": str(date.today())}]
    )
    print(f"✅ Note added for {NOTE_DATE} (id: {note_id})")
    print(f"   Total notes in collection: {notes_col.count()}")
    print(f"\nNote text:\n{note_text}")

---
## ✅ Phase 3 Complete

**What's now in place:**
- `workout_summaries` — all runs from last 6 months embedded and queryable
- `athlete_profile` — Will's background, goals, paces, coaching context embedded
- `session_notes` — persistent notes collection ready to receive post-session entries
- `memory_retrieval.py` — retrieval wrapper written to `tools/` on Drive

**Ongoing workflow:**

| Action | What to do |
|---|---|
| After a Strava/Garmin data refresh | Re-run Section 2 to rebuild workout_summaries |
| After a key session | Fill in Section 7 and run the add-note cell |
| Profile change (e.g. post-race) | Edit ATHLETE_PROFILE dict in Section 3, re-run |

**Next — Phase 4: Agents**
- `agents/feedback_agent.py` — pace evaluation, workout analysis vs targets
- `agents/recovery_agent.py` — training load monitoring, recovery advice
- `agents/planner_agent.py` — weekly schedule generation and adjustment
- `agents/coordinator_agent.py` — orchestration, routing, self-critique